# nano-dsv4.1f — build the 8K training corpus on Kaggle CPU

This notebook builds three artifacts with the already-frozen nano tokenizer:

1. **document corpus** — early / middle / late-mid causal-LM rows;
2. **reasoning traces** — Open-R1 math/science/code reasoning with integer effort 1..100;
3. **agent traces** — successful SWE trajectories plus tool/search agents, rendered with the maintained DeepSeek V4.1 protocol.

Every packed row is 8192 tokens. Late-mid documents and both trace pools retain Q-position diagnostics for the 128-query retriever budget. Trace shards also carry an assistant-only `sft_loss_mask`, so the same bytes can be used for causal midtraining or SFT.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

TOKENIZER_DATASET = 'xiayicheng3gmailcom/nano-dsv41f-tokenizer-fineweb'
REPO_URL = 'https://github.com/xiayicheng3-code/nano-dsv4.1f.git'
REPO_REF = 'codex/corpus-curriculum'
WORK = Path('/kaggle/working')
REPO_DIR = WORK / 'nano-dsv4.1f'
DOC_OUT = WORK / 'nano-dsv41f-document-corpus-8k'
TRACE_OUT = WORK / 'nano-dsv41f-traces-8k'

BUILD_DOCUMENTS = True
BUILD_TRACES = True
SEQ_LEN = 8192
TOTAL_STEPS = 10_000
QUERY_BUDGET = 128
Q_THRESHOLD = 640
Q_BANDS = '640,768,1024,1536,2048,3072,4096,6144,8192'
REASONING_TARGET_TOKENS = 4_000_000
AGENT_TARGET_TOKENS = 8_000_000
SEED = 1701

os.environ['TOKENIZERS_PARALLELISM'] = 'true'
os.environ['RAYON_NUM_THREADS'] = str(max(1, os.cpu_count() or 1))
print({'cpu_threads': os.environ['RAYON_NUM_THREADS'], 'seq_len': SEQ_LEN})


In [ ]:
# The tokenizer is a file artifact, so dataset_download is more direct than loading it into pandas.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'kagglehub', 'datasets>=3.0', 'tokenizers>=0.21', 'deepseek-recipe>=0.1'], check=True)
import kagglehub

tokenizer_root = Path(kagglehub.dataset_download(TOKENIZER_DATASET))
candidates = sorted(tokenizer_root.rglob('tokenizer.json'))
if not candidates:
    raise FileNotFoundError(f'No tokenizer.json found in {tokenizer_root}')
TOKENIZER_PATH = candidates[0]
print('tokenizer dataset:', tokenizer_root)
print('tokenizer:', TOKENIZER_PATH)


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR / 'src') + os.pathsep + env.get('PYTHONPATH', '')
print('repo commit:', commit)


In [ ]:
if BUILD_DOCUMENTS:
    if DOC_OUT.exists():
        shutil.rmtree(DOC_OUT)
    subprocess.run([
        sys.executable, str(REPO_DIR / 'scripts/prepare_document_corpus_8k.py'),
        '--tokenizer', str(TOKENIZER_PATH),
        '--output-dir', str(DOC_OUT),
        '--total-steps', str(TOTAL_STEPS),
        '--seq-len', str(SEQ_LEN),
        '--query-budget', str(QUERY_BUDGET),
        '--q-threshold', str(Q_THRESHOLD),
        '--q-band-edges', Q_BANDS,
        '--tokenize-batch-size', '256',
        '--tokenize-batch-chars', '4000000',
        '--shard-rows', '128',
        '--seed', str(SEED),
    ], check=True, cwd=REPO_DIR, env=env)


In [ ]:
if BUILD_TRACES:
    if TRACE_OUT.exists():
        shutil.rmtree(TRACE_OUT)
    subprocess.run([
        sys.executable, str(REPO_DIR / 'scripts/prepare_trace_corpus.py'),
        '--tokenizer', str(TOKENIZER_PATH),
        '--output-dir', str(TRACE_OUT),
        '--pool', 'all',
        '--seq-len', str(SEQ_LEN),
        '--reasoning-target-tokens', str(REASONING_TARGET_TOKENS),
        '--agent-target-tokens', str(AGENT_TARGET_TOKENS),
        '--query-budget', str(QUERY_BUDGET),
        '--q-threshold', str(Q_THRESHOLD),
        '--q-band-edges', Q_BANDS,
        '--tokenize-batch-size', '64',
        '--shard-rows', '128',
        '--seed', str(SEED),
    ], check=True, cwd=REPO_DIR, env=env)


In [ ]:
if BUILD_DOCUMENTS:
    curriculum = json.loads((DOC_OUT / 'curriculum_manifest.json').read_text())
    print('\nDOCUMENT CURRICULUM')
    print(json.dumps(curriculum, indent=2))
    for phase in ('early', 'middle', 'late_mid'):
        m = json.loads((DOC_OUT / phase / 'manifest.json').read_text())
        print('\n', phase)
        print(' rows:', m['packed']['rows'])
        print(' utilization:', round(m['packed']['real_token_utilization'], 4))
        print(' sources:', {k: round(v, 4) for k, v in m['phase']['source_weights_actual_real_tokens'].items()})
        print(' Q budget utilization:', round(m['query_packing']['mean_budget_utilization'], 4))
        print(' Q density:', [round(x, 6) for x in m['query_packing']['expected_q_density']])

if BUILD_TRACES:
    trace_summary = json.loads((TRACE_OUT / 'trace_manifest.json').read_text())
    print('\nTRACE CURRICULUM')
    print(json.dumps(trace_summary, indent=2))
    for pool in ('reasoning', 'agent'):
        m = json.loads((TRACE_OUT / pool / 'manifest.json').read_text())
        print('\n', pool)
        print(' records:', m['trace_records'], 'tokens:', m['actual_trace_tokens'], 'rows:', m['packing']['rows'])
        print(' assistant SFT fraction:', round(m['sft_supervised_fraction'], 4))
        print(' reasoning effort coverage:', m['reasoning_effort']['covered_values'])
        print(' Q budget utilization:', round(m['packing']['query']['mean_budget_utilization'], 4))
        print(' source final tokens:', {k: v['final_tokens'] for k, v in m['sources'].items()})
        print(' length/drop diagnostics:', {k: {kk: vv for kk, vv in v['collection'].items() if kk.startswith('dropped_')} for k, v in m['sources'].items()})


## Outputs

Keep the directories as Kaggle Dataset outputs rather than ZIP-compressing them on CPU:

- `/kaggle/working/nano-dsv41f-document-corpus-8k`
- `/kaggle/working/nano-dsv41f-traces-8k`

The NPZ shards are intentionally **uncompressed** by default: token IDs are already compact `uint16`, masks/IDs are bytes, and avoiding DEFLATE saves substantial CPU during preprocessing and training startup. Add `--compress-shards` only if Kaggle storage becomes the bottleneck.
